# MedHELM Workshop - Creating Custom Benchmarks

In this notebook, you'll learn how to:
1. Create custom prompt templates
2. Prepare custom datasets
3. Configure benchmark YAML files
4. Run custom benchmarks
5. Use LLM-as-judge evaluation

## Overview: Custom Benchmark Components

A MedHELM custom benchmark requires three files:

1. **Prompt Template (.txt)** - Task instructions with placeholders
2. **Dataset (.csv)** - Data instances with columns for the template
3. **Benchmark Config (.yaml)** - Metadata, paths, and metrics

Let's create a simple example: evaluating patient education summaries.

## 1. Setting Up

First, let's create a directory for our custom benchmark.

In [ ]:
import os

# Setup paths
HOME_DIR = os.path.expanduser("~")
CUSTOM_BENCHMARK_DIR = os.path.join(HOME_DIR, "custom-benchmarks")
OUTPUT_PATH = os.path.join(HOME_DIR, "benchmark_output")

# Create directory
os.makedirs(CUSTOM_BENCHMARK_DIR, exist_ok=True)

print(f"Custom benchmark directory: {CUSTOM_BENCHMARK_DIR}")
print(f"Output path: {OUTPUT_PATH}")

## 2. Create a Prompt Template

Let's create a prompt template for generating patient education materials.

In [ ]:
prompt_template = """You are a medical educator creating patient-friendly educational materials.

Topic: {topic}

Target Audience: {audience}

Task: Create a clear, concise patient education summary about {topic} for {audience}.

Requirements:
- Use simple, non-medical language
- Keep it under 150 words
- Include key facts and recommendations
- Be accurate and evidence-based

Your response:
"""

# Save prompt template
prompt_file = os.path.join(CUSTOM_BENCHMARK_DIR, "patient_education_prompt.txt")
with open(prompt_file, 'w') as f:
    f.write(prompt_template)

print(f"✓ Prompt template saved to: {prompt_file}")
print("\nContent:")
print(prompt_template)

## 3. Create a Custom Dataset

Now let's create a CSV dataset with instances for our benchmark.

In [ ]:
import csv

# Define sample dataset
dataset = [
    {
        "topic": "Type 2 Diabetes",
        "audience": "newly diagnosed adults",
        "correct_answer": "Type 2 diabetes is a condition where your body doesn't use insulin properly. This causes high blood sugar levels. You can manage it through healthy eating, regular exercise, and medication if needed. Check your blood sugar regularly, maintain a healthy weight, and follow your doctor's advice. With proper management, you can live a healthy, active life.",
        "incorrect_answers": '["Diabetes is not serious", "You cannot control diabetes"]'
    },
    {
        "topic": "Hypertension",
        "audience": "older adults",
        "correct_answer": "High blood pressure (hypertension) means your heart works harder to pump blood. It often has no symptoms but can lead to serious problems. Control it by eating less salt, staying active, taking prescribed medicines, and monitoring your blood pressure at home. Regular check-ups are important. Small lifestyle changes can make a big difference.",
        "incorrect_answers": '["High blood pressure is normal with age", "Medication is not necessary"]'
    },
    {
        "topic": "Asthma",
        "audience": "parents of children with asthma",
        "correct_answer": "Asthma causes airways to narrow, making breathing difficult. Triggers include allergies, exercise, and infections. Use prescribed inhalers as directed - controller inhalers daily and rescue inhalers for flare-ups. Create an asthma action plan with your doctor. Avoid triggers, keep track of symptoms, and seek emergency care if symptoms worsen despite medication.",
        "incorrect_answers": '["Children will outgrow asthma without treatment", "Inhalers are addictive"]'
    }
]

# Save dataset as CSV
dataset_file = os.path.join(CUSTOM_BENCHMARK_DIR, "patient_education_dataset.csv")
with open(dataset_file, 'w', newline='') as f:
    if dataset:
        writer = csv.DictWriter(f, fieldnames=dataset[0].keys())
        writer.writeheader()
        writer.writerows(dataset)

print(f"✓ Dataset saved to: {dataset_file}")
print(f"\nDataset contains {len(dataset)} instances:")
for i, item in enumerate(dataset, 1):
    print(f"  {i}. {item['topic']} for {item['audience']}")

## 4. Create Benchmark Configuration

Now let's create the YAML configuration file that ties everything together.

In [ ]:
import yaml

# Define benchmark configuration
config = {
    'name': 'PATIENT-EDUCATION-DEMO',
    'description': 'Evaluates LLMs on creating patient-friendly educational materials about common medical conditions.',
    'prompt_file': 'patient_education_prompt.txt',
    'dataset_file': 'patient_education_dataset.csv',
    'max_tokens': 200,
    'metrics': [
        {
            'name': 'rouge_l'
        },
        {
            'name': 'BERTScore-F'
        }
    ]
}

# Save configuration
config_file = os.path.join(CUSTOM_BENCHMARK_DIR, "patient_education_config.yaml")
with open(config_file, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print(f"✓ Configuration saved to: {config_file}")
print("\nConfiguration:")
print(yaml.dump(config, default_flow_style=False))

## 5. Create Run Configuration

Create a run configuration file (.conf) to specify which models to evaluate.

In [ ]:
# Create run configuration
# Note: Adjust model and model_deployment based on your setup
run_config = f"""entries: [
  {{description: "medhelm_configurable_benchmark:model=openai/gpt-3.5-turbo,config_path={config_file}", priority: 1}},
]
"""

run_config_file = os.path.join(CUSTOM_BENCHMARK_DIR, "patient_education_run.conf")
with open(run_config_file, 'w') as f:
    f.write(run_config)

print(f"✓ Run configuration saved to: {run_config_file}")
print("\nContent:")
print(run_config)

## 6. Running the Custom Benchmark

Now let's run our custom benchmark!

In [ ]:
import subprocess

SUITE_NAME = "custom-patient-education"

# Build command
cmd = [
    "helm-run",
    "--conf-paths", run_config_file,
    "--suite", SUITE_NAME,
    "--max-eval-instances", "3",  # Run all 3 instances
    "--output-path", OUTPUT_PATH
]

print("Running custom benchmark...")
print(" ".join(cmd))
print("\n" + "="*60)

try:
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=600)
    print(result.stdout)
    if result.stderr:
        print("\nSTDERR:")
        print(result.stderr)
    
    if result.returncode == 0:
        print("\n✓ Custom benchmark completed successfully!")
    else:
        print(f"\n✗ Benchmark failed with return code {result.returncode}")
except subprocess.TimeoutExpired:
    print("\n✗ Benchmark timed out")
except FileNotFoundError:
    print("\n✗ helm-run not found. Ensure the environment is activated.")
except Exception as e:
    print(f"\n✗ Error: {str(e)}")

## 7. Summarize Results

Create a summary with auto-generated schema.

In [ ]:
import subprocess

# Run helm-summarize with auto-generate-schema
cmd = [
    "helm-summarize",
    "--auto-generate-schema",
    "--suite", SUITE_NAME,
    "--output-path", OUTPUT_PATH
]

print("Creating summary...")
print(" ".join(cmd))
print()

try:
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    
    if result.returncode == 0:
        print("\n✓ Summary created successfully!")
        print(f"\nTo view results, run:")
        print(f"helm-server --suite {SUITE_NAME} --output-path {OUTPUT_PATH}")
    else:
        print(f"\n✗ Summarization failed with return code {result.returncode}")
except Exception as e:
    print(f"\n✗ Error: {str(e)}")

## 8. Advanced: LLM-as-Judge Evaluation

For more nuanced evaluation, you can use LLM-as-judge with custom rubrics.

In [ ]:
# Create judge prompt for LLM-as-judge evaluation
judge_prompt = """You are an expert in patient education and health communication.

Original Task:
<QUESTION>
{QUESTION}
</QUESTION>

Model Response:
<RESPONSE>
{RESPONSE}
</RESPONSE>

Reference Answer:
<GOLD_RESPONSE>
{GOLD_RESPONSE}
</GOLD_RESPONSE>

Evaluate the model's response on these criteria (score 0-5 for each):

1. **Clarity** (0 = incomprehensible; 5 = crystal clear for target audience)
2. **Accuracy** (0 = medically incorrect; 5 = medically accurate)
3. **Completeness** (0 = missing key information; 5 = covers all important points)
4. **Accessibility** (0 = uses complex jargon; 5 = appropriate language level)
5. **Actionability** (0 = no practical guidance; 5 = clear action steps)

Output Format:
Provide your evaluation as valid JSON inside <rubric_criteria> tags:

<rubric_criteria>
{
  "clarity": {
    "score": 0,
    "explanation": "Explain your reasoning."
  },
  "accuracy": {
    "score": 0,
    "explanation": "Explain your reasoning."
  },
  "completeness": {
    "score": 0,
    "explanation": "Explain your reasoning."
  },
  "accessibility": {
    "score": 0,
    "explanation": "Explain your reasoning."
  },
  "actionability": {
    "score": 0,
    "explanation": "Explain your reasoning."
  }
}
</rubric_criteria>

Ensure valid JSON: use double quotes, escape internal quotes, no trailing commas.
"""

# Save judge prompt
judge_prompt_file = os.path.join(CUSTOM_BENCHMARK_DIR, "patient_education_judge_prompt.txt")
with open(judge_prompt_file, 'w') as f:
    f.write(judge_prompt)

print(f"✓ Judge prompt saved to: {judge_prompt_file}")
print("\nThis can be used in a benchmark config with jury_score metric.")

## 9. Best Practices for Custom Benchmarks

### Prompt Templates:
- Use clear, specific instructions
- Include task constraints (length, format, etc.)
- Use {column_name} placeholders that match your CSV

### Datasets:
- Include diverse, representative examples
- Always provide `correct_answer` column
- Provide `incorrect_answers` as JSON list
- Test with a small subset first

### Metrics:
- Choose appropriate metrics for your task
- Consider using multiple complementary metrics
- For open-ended tasks, consider LLM-as-judge

### Validation:
- Start with max-eval-instances=1 to test
- Verify prompt rendering looks correct
- Check that metrics align with your goals

## Summary

In this notebook, you learned:
1. ✓ How to create custom prompt templates
2. ✓ How to prepare custom datasets in CSV format
3. ✓ How to configure benchmark YAML files
4. ✓ How to run custom benchmarks
5. ✓ How to use LLM-as-judge evaluation
6. ✓ Best practices for custom benchmark creation

### Next Steps:

- Create benchmarks for your specific use cases
- Experiment with different metrics
- Build LLM-as-judge rubrics for complex evaluations
- Share your benchmarks with the community